# 01. MBMP baseline on three Turkmenistan super-emitter sites

Multi-band multi-pass (MBMP) methane enhancement (Varon et al. 2021) on Sentinel-2 L1C
(`COPERNICUS/S2_HARMONIZED`) chips for CH4Net sites **T7, T14, T22** in 2021.

**Sign convention.** dR = (c * R12 - R11) / R11, so methane plumes are **negative**.
Multi-pass = target dR minus dR of the most recent clear scene in the prior 30 days.

**Known plume dates.** CH4Net (Vaughan et al. 2024) hand labels mark which 2021 images contain a plume,
but the release has no dates. `plumescout.ch4net_index` recovers each image's date by matching it to our
chips (cloud patterns, band texture and the B12/B11 ratio, solved as a one-to-one assignment).
`conf` is the margin over the next best date. Dates with low `conf` are uncertain, and this notebook
reports `conf` next to every result.

**Caveat.** The B12/B11 ratio used for date matching also carries the plume signal, so a date can be
pulled towards one where our ratio image looks plume-like. Control dates (CH4Net images with no plume,
matched the same way) are shown for comparison.

In [1]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from plumescout import mbmp
from plumescout.ch4net_index import place_centred
from plumescout.config import load_yaml, resolve_path

logging.basicConfig(level=logging.WARNING)
cfg = load_yaml("sites.yaml")
SITES = cfg["phase1_sites"]
K = cfg["mbmp"]["threshold_k"]
MAX_CLOUD = cfg["sentinel2"]["max_cloud_frac_clear"]
BANDS = cfg["sentinel2"]["bands"]
SCALE = cfg["s2_chip"]["dn_scale"]
N_DATES = 5
scenes = pd.read_parquet(resolve_path(cfg["paths"]["scenes_table"]))
index = pd.read_parquet(resolve_path("data/ch4net_index.parquet"))
CH4 = resolve_path("data/raw/ch4net/test")
n_sites = index.site_id.nunique()
print(f"scenes: {len(scenes)} chips; CH4Net index: {len(index)} images, {n_sites} sites")

scenes: 316 chips; CH4Net index: 285 images, 3 sites


In [2]:
def read_refl(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float64) / SCALE  # (13, H, W) TOA reflectance


def b(a, name):
    return a[BANDS.index(name)]


def rgb(a):
    x = np.stack([b(a, "B4"), b(a, "B3"), b(a, "B2")], -1)
    lo, hi = np.nanpercentile(x, [2, 98])
    return np.clip((x - lo) / (hi - lo), 0, 1)


def scene_row(site, date):
    return scenes[(scenes.site_id == site) & (scenes.date == date)].iloc[0]


def multi_pass_for(site, date):
    row = scene_row(site, date)
    ref = scene_row(site, row.ref_scene_date)
    t, r = read_refl(row.chip_path), read_refl(ref.chip_path)
    dr = mbmp.multi_pass(b(t, "B11"), b(t, "B12"), b(r, "B11"), b(r, "B12"))
    return t, dr, row.ref_scene_date

## Pick dates

Per site: the `N_DATES` CH4Net plume images with the highest date-match confidence, restricted to clear
(< 20% cloud plus cloud shadow) dates that have a clear reference scene. Controls: the same number of CH4Net no-plume images.

In [3]:
cols = ["site_id", "date", "ref_scene_date", "obscured_frac"]
ok = scenes[cols].rename(columns={"obscured_frac": "chip_obscured"})
cand = index.merge(ok, on=["site_id", "date"])
cand = cand[cand.ref_scene_date.notna() & (cand.chip_obscured < MAX_CLOUD)]
pick = []
for site in SITES:
    c = cand[cand.site_id == site].sort_values("conf", ascending=False)
    pick.append(c[c.plume_px > 0].drop_duplicates("date").head(N_DATES).assign(kind="plume"))
    pick.append(c[c.plume_px == 0].drop_duplicates("date").head(N_DATES).assign(kind="control"))
pick = pd.concat(pick, ignore_index=True)
pick[["site_id", "kind", "ch4net_id", "date", "ref_scene_date", "plume_px", "conf"]]

,site_id,kind,ch4net_id,date,ref_scene_date,plume_px,conf
0,T7,plume,1210,2021-04-12,2021-04-09,507,0.272773
1,T7,plume,1191,2021-08-02,2021-07-31,577,0.221506
2,T7,plume,1209,2021-04-14,2021-04-12,361,0.174645
3,T7,plume,1196,2021-10-29,2021-10-24,1551,0.170097
4,T7,plume,1216,2021-08-12,2021-08-10,915,0.170065
5,T7,control,1108,2021-03-15,2021-03-10,0,0.773262
6,T7,control,1097,2021-02-26,2021-02-23,0,0.698657
7,T7,control,1126,2021-04-04,2021-03-30,0,0.658373
8,T7,control,1123,2021-12-08,2021-12-05,0,0.657127
9,T7,control,1136,2021-01-22,2021-01-07,0,0.606012


## Does dR go negative where CH4Net labelled a plume?

For each picked image: median multi-pass dR inside the CH4Net label (placed centre to centre on our grid)
and outside it, the share of labelled pixels caught by the k-sigma mask, and the share of unlabelled
pixels flagged. For controls there is no label, so only the flagged share is meaningful.

In [4]:
rows, cache = [], {}
for r in pick.itertuples():
    t, dr, ref_date = multi_pass_for(r.site_id, r.date)
    mask = mbmp.threshold_mask(dr, k=K)
    lab = place_centred(np.load(CH4 / f"label/{r.ch4net_id}.npy") > 0, dr.shape)
    cache[(r.site_id, r.kind, r.date)] = (t, dr, mask, lab, ref_date)
    inside = dr[lab & np.isfinite(dr)]
    outside = dr[~lab & np.isfinite(dr)]
    rows.append(
        {
            "site_id": r.site_id,
            "kind": r.kind,
            "date": r.date,
            "conf": round(r.conf, 3),
            "label_px_on_chip": int(lab.sum()),
            "median_dR_label": round(float(np.median(inside)), 4) if inside.size else np.nan,
            "median_dR_outside": round(float(np.median(outside)), 4),
            "mask_recall_in_label": round(float(mask[lab].mean()), 3) if lab.any() else np.nan,
            "mask_frac_outside": round(float(mask[~lab].mean()), 4),
        }
    )
res = pd.DataFrame(rows)
res

,site_id,kind,date,conf,label_px_on_chip,median_dR_label,median_dR_outside,mask_recall_in_label,mask_frac_outside
0,T7,plume,2021-04-12,0.273,507,-0.0040,0.0000,0.158,0.0143
1,T7,plume,2021-08-02,0.222,577,0.0032,0.0001,0.031,0.0138
2,T7,plume,2021-04-14,0.175,361,-0.0010,0.0003,0.000,0.0081
3,T7,plume,2021-10-29,0.170,1551,-0.0103,-0.0004,0.036,0.0074
4,T7,plume,2021-08-12,0.170,915,-0.0039,-0.0001,0.122,0.0087
5,T7,control,2021-03-15,0.773,0,NaN,0.0004,NaN,0.0088
6,T7,control,2021-02-26,0.699,0,NaN,0.0048,NaN,0.0025
7,T7,control,2021-04-04,0.658,0,NaN,-0.0033,NaN,0.0060
8,T7,control,2021-12-08,0.657,0,NaN,0.0013,NaN,0.0295
9,T7,control,2021-01-22,0.606,0,NaN,-0.0018,NaN,0.0129


In [5]:
summary = res.groupby(["site_id", "kind"]).agg(
    n_dates=("date", "count"),
    median_dR_label=("median_dR_label", "median"),
    median_dR_outside=("median_dR_outside", "median"),
    label_more_negative=(
        "median_dR_label",
        lambda s: int((s < res.loc[s.index, "median_dR_outside"]).sum()),
    ),
    mask_recall_in_label=("mask_recall_in_label", "median"),
    mask_frac_outside=("mask_frac_outside", "median"),
)
summary

n_dates  median_dR_label  median_dR_outside  \
site_id kind                                                   
T14     control        5              NaN             0.0018   
        plume          5          -0.0075             0.0005   
T22     control        5              NaN            -0.0013   
        plume          5          -0.0177             0.0011   
T7      control        5              NaN             0.0004   
        plume          5          -0.0039             0.0000   

                 label_more_negative  mask_recall_in_label  mask_frac_outside  
site_id kind                                                                   
T14     control                    0                   NaN             0.0505  
        plume                      5                 0.060             0.0095  
T22     control                    0                   NaN             0.0199  
        plume                      5                 0.157             0.0098  
T7      control                    0                   NaN             0.0088  
        plume                      4                 0.036             0.0087

## Figures

One figure per site: RGB, multi-pass dR (red = negative = methane-like) with the CH4Net label outlined,
and the k-sigma mask. Plume dates first, then two control dates.

In [6]:
LIM = 0.08
paths = []
for site in SITES:
    keys = [k for k in cache if k[0] == site and k[1] == "plume"]
    keys += [k for k in cache if k[0] == site and k[1] == "control"][:2]
    fig, axes = plt.subplots(
        len(keys), 3, figsize=(9, 3 * len(keys)), squeeze=False, layout="constrained"
    )
    for ax_row, key in zip(axes, keys, strict=True):
        t, dr, mask, lab, ref_date = cache[key]
        conf = float(
            pick[(pick.site_id == site) & (pick.kind == key[1]) & (pick.date == key[2])].conf.iloc[
                0
            ]
        )
        ax_row[0].imshow(rgb(t))
        ax_row[0].set_ylabel(f"{key[2]} ({key[1]})\nref {ref_date}, conf {conf:.2f}", fontsize=8)
        im = ax_row[1].imshow(dr, cmap="RdBu", vmin=-LIM, vmax=LIM)
        if lab.any():
            ax_row[1].contour(lab, levels=[0.5], colors="k", linewidths=0.8)
        ax_row[2].imshow(mask, cmap="gray_r")
        for a in ax_row:
            a.set_xticks([])
            a.set_yticks([])
    axes[0, 0].set_title("RGB (B4, B3, B2)")
    axes[0, 1].set_title("multi-pass dR, CH4Net label")
    axes[0, 2].set_title(f"mask, k = {K}")
    fig.colorbar(im, ax=axes, location="bottom", shrink=0.5, label="dR (negative = CH4 absorption)")
    fig.suptitle(f"CH4Net site {site}, Sentinel-2 L1C, 2021. 2 km x 2 km chips, 10 m pixels")
    p = resolve_path(f"figures/01_mbmp_{site}.png")
    p.parent.mkdir(exist_ok=True)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    paths.append(p.name)
    plt.close(fig)
paths

['01_mbmp_T7.png', '01_mbmp_T14.png', '01_mbmp_T22.png']

In [7]:
for site in SITES:
    s = res[res.site_id == site]
    p, c = s[s.kind == "plume"], s[s.kind == "control"]
    print(
        f"{site}: plume dates {len(p)}, label more negative than background on "
        f"{int((p.median_dR_label < p.median_dR_outside).sum())}/{len(p)}; "
        f"median dR in label {p.median_dR_label.median():+.4f} "
        f"vs outside {p.median_dR_outside.median():+.4f}; "
        f"mask recall in label {p.mask_recall_in_label.median():.2f}; "
        f"control mask frac {c.mask_frac_outside.median():.4f}; "
        f"date-match conf range {p.conf.min():.2f} to {p.conf.max():.2f}"
    )

T7: plume dates 5, label more negative than background on 4/5; median dR in label -0.0039 vs outside +0.0000; mask recall in label 0.04; control mask frac 0.0088; date-match conf range 0.17 to 0.27
T14: plume dates 5, label more negative than background on 5/5; median dR in label -0.0075 vs outside +0.0005; mask recall in label 0.06; control mask frac 0.0505; date-match conf range 0.20 to 0.54
T22: plume dates 5, label more negative than background on 5/5; median dR in label -0.0177 vs outside +0.0011; mask recall in label 0.16; control mask frac 0.0199; date-match conf range 0.02 to 0.05
